In [2]:
import numpy as np
import pandas as pd

In [3]:
train_trans = pd.read_csv('../data/raw/train_transaction.csv')
train_id = pd.read_csv('../data/raw/train_identity.csv')


In [6]:
train = train_trans.merge(train_id, how='left', on='TransactionID')
print(train.shape)
print(f'{train.memory_usage(index=True, deep=True).sum():,}')

(590540, 434)
2,636,093,316


In [13]:
target = train_trans['isFraud']
print(target.value_counts(normalize=True))


isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [6]:
trans_amt = train_trans['TransactionAmt']
print(trans_amt.describe())

count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64


In [7]:
product_cat = train_trans['ProductCD']
print(product_cat.value_counts())

ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64


In [9]:
c4 = train_trans['card4']
print(c4.value_counts())
print(c4.isnull().sum())
print(c4.duplicated())

card4
visa                384767
mastercard          189217
american express      8328
discover              6651
Name: count, dtype: int64


In [3]:
train_trans_len = len(train_trans)
print(train_trans_len)

590540


In [9]:
null_train_trans = pd.DataFrame({
    'null_count': train_trans.isnull().sum(),
    'null_pct': ((train_trans.isnull().sum()/train_trans_len) * 100).round(2)
}).sort_values('null_pct', ascending=False)
pd.set_option('display.max_rows', None)
print(null_train_trans[null_train_trans['null_count'] > 0])

               null_count  null_pct
dist2              552913     93.63
D7                 551623     93.41
D13                528588     89.51
D14                528353     89.47
D12                525823     89.04
D6                 517353     87.61
D9                 515614     87.31
D8                 515614     87.31
V153               508595     86.12
V140               508595     86.12
V150               508589     86.12
V149               508595     86.12
V155               508595     86.12
V147               508595     86.12
V146               508595     86.12
V145               508589     86.12
V144               508589     86.12
V143               508589     86.12
V142               508595     86.12
V141               508595     86.12
V152               508589     86.12
V157               508595     86.12
V139               508595     86.12
V138               508595     86.12
V154               508595     86.12
V151               508589     86.12
V165               508589   

In [11]:
zero_null_cols = train_trans.columns[train_trans.isnull().sum() == 0].tolist()
print(zero_null_cols)
print(train_trans[zero_null_cols].dtypes.value_counts())

['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
float64    15
int64       4
str         1
Name: count, dtype: int64


In [12]:
int_cols = train_trans[zero_null_cols].select_dtypes(include=['int64']).columns.tolist()
print(train_trans[int_cols].agg(['min', 'max']))

     TransactionID  isFraud  TransactionDT  card1
min        2987000        0          86400   1000
max        3577539        1       15811131  18396


In [13]:
float_cols = train_trans[zero_null_cols].select_dtypes(include=['float64']).columns.tolist()
print(train_trans[float_cols].agg(['min', 'max']))

     TransactionAmt      C1      C2    C3      C4     C5      C6      C7  \
min           0.251     0.0     0.0   0.0     0.0    0.0     0.0     0.0   
max       31937.391  4685.0  5691.0  26.0  2253.0  349.0  2253.0  2255.0   

         C8     C9     C10     C11     C12     C13     C14  
min     0.0    0.0     0.0     0.0     0.0     0.0     0.0  
max  3331.0  210.0  3257.0  3188.0  3188.0  2918.0  1429.0  


In [3]:
print((train_trans['TransactionAmt'] % 1 == 0).all())

False


In [4]:
train_trans['isFraud'] = train_trans['isFraud'].astype(np.uint8)
train_trans['card1'] = train_trans['card1'].astype(np.uint16)
train_trans['TransactionDT'] = train_trans['TransactionDT'].astype(np.uint32)
train_trans['TransactionID'] = train_trans['TransactionID'].astype(np.uint32)

In [5]:
train_trans['TransactionAmt'] = train_trans['TransactionAmt'].astype(np.float32)

In [6]:
train_trans['C1'] = train_trans['C1'].astype(np.uint16)
train_trans['C2'] = train_trans['C2'].astype(np.uint16)
train_trans['C3'] = train_trans['C3'].astype(np.uint8)
train_trans['C4'] = train_trans['C4'].astype(np.uint16)
train_trans['C5'] = train_trans['C5'].astype(np.uint16)
train_trans['C6'] = train_trans['C6'].astype(np.uint16)
train_trans['C7'] = train_trans['C7'].astype(np.uint16)
train_trans['C8'] = train_trans['C8'].astype(np.uint16)
train_trans['C9'] = train_trans['C9'].astype(np.uint8)
train_trans['C10'] = train_trans['C10'].astype(np.uint16)
train_trans['C11'] = train_trans['C11'].astype(np.uint16)
train_trans['C12'] = train_trans['C12'].astype(np.uint16)
train_trans['C13'] = train_trans['C13'].astype(np.uint16)
train_trans['C14'] = train_trans['C14'].astype(np.uint16)

In [11]:
original = train_trans['TransactionAmt'].astype(np.float64)
converted = train_trans['TransactionAmt'].astype(np.float32)
diff = (original - converted).abs()
print(f"Max precision loss: {diff.max()}")
print(f"Mean precision loss: {diff.mean()}")

Max precision loss: 0.0
Mean precision loss: 0.0


In [12]:
float64_cols = train_trans.select_dtypes(include='float64').columns.tolist()
print(float64_cols)

['card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V88', 'V89', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95', 'V96', 'V97', 'V98', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V11

In [13]:
train_trans[float64_cols] = train_trans[float64_cols].astype(np.float32)

In [14]:
after = train_trans.memory_usage(deep=True).sum() / 1e6
print(f"After: {after:.2f} MB")

After: 1243.95 MB


In [4]:
null_train_id = pd.DataFrame({
    'null_count': train_id.isnull().sum(),
    'null_pct': ((train_id.isnull().sum()/len(train_id)) * 100).round(2)
}).sort_values('null_pct', ascending=False)
pd.set_option('display.max_rows', None)
print(null_train_id[null_train_id['null_count'] > 0])

            null_count  null_pct
id_24           139486     96.71
id_25           139101     96.44
id_07           139078     96.43
id_08           139078     96.43
id_21           139074     96.42
id_22           139064     96.42
id_23           139064     96.42
id_27           139064     96.42
id_26           139070     96.42
id_18            99120     68.72
id_04            77909     54.02
id_03            77909     54.02
id_33            70944     49.19
id_10            69307     48.05
id_09            69307     48.05
id_30            66668     46.22
id_32            66647     46.21
id_34            66428     46.06
id_14            64189     44.50
DeviceInfo       25567     17.73
id_13            16913     11.73
id_16            14893     10.33
id_06             7368      5.11
id_05             7368      5.11
id_20             4972      3.45
id_19             4915      3.41
id_17             4864      3.37
id_31             3951      2.74
DeviceType        3423      2.37
id_02     

In [7]:
no_null_cols1=  train_id.columns[train_id.isnull().sum() == 0].tolist()
print(no_null_cols1)
print(train_id[no_null_cols1].dtypes.value_counts())

['TransactionID', 'id_01', 'id_12']
int64      1
float64    1
str        1
Name: count, dtype: int64


In [8]:
int_cols1 = train_id[no_null_cols1].select_dtypes(include=['int64']).columns.tolist()
print(train_id[int_cols1].agg(['min', 'max']))

     TransactionID
min        2987004
max        3577534


In [10]:
id01 = train_id['id_01']
print(id01.value_counts(normalize=False))


id_01
-5.0      82170
 0.0      19555
-10.0     11257
-20.0     11211
-15.0      5674
-25.0      4623
-45.0      2143
-35.0      1622
-40.0      1385
-100.0     1012
-50.0       709
-30.0       682
-95.0       428
-60.0       410
-55.0       320
-80.0       220
-90.0       214
-70.0        97
-65.0        93
-85.0        87
-75.0        83
-18.0        23
-12.0        15
-11.0        15
-6.0         15
-16.0        13
-21.0        12
-7.0         10
-14.0        10
-19.0         9
-31.0         9
-17.0         9
-26.0         8
-27.0         6
-87.0         5
-23.0         5
-22.0         5
-37.0         5
-9.0          4
-62.0         4
-13.0         4
-44.0         3
-96.0         3
-38.0         3
-64.0         2
-99.0         2
-53.0         2
-88.0         2
-61.0         2
-46.0         2
-29.0         2
-8.0          2
-71.0         2
-56.0         2
-72.0         1
-58.0         1
-42.0         1
-76.0         1
-34.0         1
-54.0         1
-28.0         1
-94.0         1
-2

In [11]:
print(id01.min(), id01.max())

-100.0 0.0


In [12]:
train_id['id_01'] = train_id['id_01'].astype(np.int8)
train_id['TransactionID'] = train_id['TransactionID'].astype(np.int32)

In [13]:
float64_cols1 = train_id.select_dtypes(include='float64').columns.tolist()
train_id[float64_cols1] = train_id[float64_cols1].astype(np.float32)

In [18]:
original1 = train_id['id_01'].astype(np.float64)
converted1 = train_id['id_01'].astype(np.float32)
diff = (original1 - converted1).abs()
print(f"Max precision loss: {diff.max()}")
print(f"Mean precision loss: {diff.mean()}")

Max precision loss: 0.0
Mean precision loss: 0.0


In [14]:
after_id = train_id.memory_usage(deep=True).sum() / 1e6
print(f"train identity after: {after_id:.2f} MB")

train identity after: 135.82 MB


In [19]:
train = train_trans.merge(train_id, how='left', on='TransactionID')
print(train.shape)
print(f'{train.memory_usage(index=True, deep=True).sum():,}')

(590540, 434)
2,584,125,796


In [20]:
obj_cols = train.select_dtypes(include='object').columns.tolist()
print(obj_cols)
print(f"Object cols memory: {train[obj_cols].memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Non-object memory: {train.drop(columns=obj_cols).memory_usage(deep=True).sum() / 1e6:.1f} MB")

C:\Users\moizs\AppData\Local\Temp\ipykernel_3408\18688683.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = train.select_dtypes(include='object').columns.tolist()


['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
Object cols memory: 732.2 MB
Non-object memory: 1851.9 MB


In [21]:
for col in obj_cols:
    train[col] = train[col].astype('category')

print(f"After category encoding: {train.memory_usage(deep=True).sum() / 1e6:.1f} MB")

After category encoding: 1872.2 MB
